```sh
wget http://www.fredjo.com/files/ihdp_npci_1-1000.train.npz.zip
wget http://www.fredjo.com/files/ihdp_npci_1-1000.test.npz.zip
```

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
from econml.dml import CausalForestDML, LinearDML
from xgboost import XGBClassifier, XGBRegressor
from causal_parameters import CausalParameters
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
from utils import print_metrics
from plot_propensity_scores import plot_propensity_scores
from plot_outcome_nuisance import plot_outcome_nuisance

In [3]:
data_path = Path("datasets/ihdp")
data_train = np.load(data_path / "ihdp_npci_1-1000.train.npz")

In [4]:
x = data_train["x"]      # Shape: (747, 25, 1000) -> 747 samples, 25 features, 1000 runs
t = data_train["t"]      # Shape: (747, 1000)
yf = data_train["yf"]    # Shape: (747, 1000)

In [5]:
def build_df(data):

    # Select a specific realization run (0 to 999)
    run_idx = 0
    
    # Actual data
    x_mat = data['x'][:, :, run_idx]  # Confounding variables

    # Simulated data (although it would be available)
    t_vec = data['t'][:, run_idx]    # Treatment administered (binary)
    yf_vec = data['yf'][:, run_idx]  # Factual outcome

    # Simulated data (unobservable)    
    ycf_vec = data['ycf'][:, run_idx]  # Counterfactual outcome
    mu0_vec = data['mu0'][:, run_idx]  # Expected baseline outcome for an individual with covariates x if they do not receive the treatment
    mu1_vec = data['mu1'][:, run_idx]  # Expected baseline outcome for an individual with covariates x if they do receive the treatment

    # y(0) = mu0 + noise
    # y(1) = mu1 + noise
    # mu1(x) = mu0(x) + tau(x)
    
    # Create feature column names (x1 to x25)
    feature_cols = [f"x{i}" for i in range(1, 26)]
    
    # Construct DataFrame for the covariates
    df = pd.DataFrame(x_mat, columns=feature_cols)
    
    # Add causal indicators and outcomes
    df['treatment'] = t_vec.astype(int)
    df['y_factual'] = yf_vec
    df['y_counterfactual'] = ycf_vec
    df['mu0'] = mu0_vec
    df['mu1'] = mu1_vec
    
    # Calculate true CATE ground truth for validation
    df['true_cate'] = df['mu1'] - df['mu0']

    # x1, x2, ..., x25
    feature_columns = [f"x{i}" for i in range(1, 26)]
    
    return df, feature_columns

In [6]:
df, feature_columns = build_df(data=data_train)    

In [7]:
df.head().round(2)

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,...,x22,x23,x24,x25,treatment,y_factual,y_counterfactual,mu0,mu1,true_cate
0,1.40,1.0,-1.11,-0.88,0.31,-1.02,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1,4.77,-0.30,1.16,5.82,4.66
1,0.27,0.2,0.38,0.16,-0.63,1.46,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0,2.96,5.78,3.49,6.92,3.43
2,1.05,1.8,-1.11,0.16,-0.63,0.96,1.0,0.0,1.0,1.0,...,0.0,0.0,0.0,1.0,0,4.16,7.06,3.16,6.82,3.66
3,0.66,0.2,-0.73,-0.88,0.37,-0.69,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1,6.17,1.38,1.47,6.06,4.59
4,0.86,1.8,0.01,-0.88,0.56,0.30,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1,7.83,2.75,2.19,6.45,4.27


In [8]:
df.describe().round(2)

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,...,x22,x23,x24,x25,treatment,y_factual,y_counterfactual,mu0,mu1,true_cate
count,672.00,672.00,672.00,672.00,672.00,672.00,672.0,672.00,672.00,672.00,...,672.00,672.00,672.00,672.00,672.00,672.00,672.00,672.00,672.00,672.00
mean,0.01,0.01,-0.00,-0.01,-0.01,0.01,0.5,0.10,0.52,0.36,...,0.08,0.08,0.12,0.16,0.19,3.15,5.70,2.44,6.45,4.01
std,1.00,1.00,1.01,0.99,1.00,1.01,0.5,0.29,0.50,0.48,...,0.28,0.27,0.33,0.37,0.39,2.17,1.97,1.27,0.45,0.85
min,-2.73,-3.80,-1.85,-0.88,-5.13,-1.85,0.0,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,-1.54,-1.04,0.92,5.59,-1.87
25%,-0.64,-0.60,-0.73,-0.88,-0.57,-0.86,0.0,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,1.64,5.05,1.52,6.09,3.74
50%,0.16,0.20,-0.36,0.16,0.06,-0.03,1.0,0.00,1.00,0.00,...,0.00,0.00,0.00,0.00,0.00,2.58,6.21,2.13,6.43,4.29
75%,0.84,0.60,0.76,0.16,0.62,0.63,1.0,0.00,1.00,1.00,...,0.00,0.00,0.00,0.00,0.00,4.45,6.93,3.05,6.78,4.57
max,1.51,2.60,2.99,2.24,2.37,2.95,1.0,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.00,11.27,10.17,9.82,7.95,4.67


In [9]:
def evaluate_outcome_nuisance(
    model_y,
    X: pd.DataFrame,
    Y: pd.Series,
    treatment: pd.Series,
    cv_splitter,
) -> dict[str, float]:
    """Evaluates the outcome nuisance model m(X) = E[Y | X] prior to DML."""
    # Compute out-of-fold predictions for Y using CV
    y_pred = cross_val_predict(model_y, pd.concat([X, treatment], axis=1), Y.to_numpy(), cv=cv_splitter)
    residuals = Y.to_numpy() - y_pred

    # 1. Global Performance Metrics
    rmse = float(np.sqrt(mean_squared_error(Y, y_pred)))
    r2 = float(r2_score(Y, y_pred))

    # 2. Residual Mean by Treatment Group
    res_treated_mean = float(np.mean(residuals[treatment == 1]))
    res_control_mean = float(np.mean(residuals[treatment == 0]))

    metrics = {
        "out_of_fold_rmse": rmse,
        "out_of_fold_r2": r2,
        "residual_mean_treated": res_treated_mean,
        "residual_mean_control": res_control_mean,
    }

    plot_outcome_nuisance(y_pred, Y, treatment, residuals, r2, rmse)
    

    return metrics

In [10]:
def double_ml_econml_inline(
    df: pd.DataFrame,
    feature_columns: list[str],
    treatment_column: str = "treatment",        
    outcome_column: str = "outcome",    
    ) -> CausalParameters:
    
    X = df[feature_columns]
    T = df[treatment_column]
    Y = df[outcome_column]

    cv_splitter = KFold(n_splits=10, shuffle=True, random_state=0)
    
    # Define base machine learning model for outcome nuisance: m(X) = E[Y | X]
    model_y = XGBRegressor(
        n_estimators=1000,
        max_depth=2,
        learning_rate=0.05,
        random_state=0
        )

    # ---> Nuisance from model y: outcome nuisance
    # Expected outcome given covariates and treatment
    evaluate_outcome_nuisance(model_y, X, Y, T, cv_splitter)
    
    # Define base machine learning model for propensity nuisance: e(X) = P(T=1 | X)
    model_t = XGBClassifier(
        n_estimators=1000,
        max_depth=2,
        learning_rate=0.05,
        random_state=0,
        eval_metric="logloss",
        )

    # ---> Nuisance from model t: Propensity score
    # Probability of receiving treatment given covariates
    propensity_scores = cross_val_predict(
        model_t,
        X,
        T.to_numpy(),
        cv=cv_splitter,
        method="predict_proba"
    )[:, 1]

    plot_propensity_scores(
        df=df,
        propensity_scores=propensity_scores
    )
    
    # Initialize the double machine learning estimator
    model = LinearDML(
        model_y=model_y,
        model_t=model_t,
        discrete_treatment=True,
        cv=cv_splitter,
        random_state=0,
        )

    # Fit the double ML model on observational data [X, T, Y] (EconML expects Y and T as 1D arrays or column vectors)
    model.fit(Y=Y.to_numpy(), T=T.to_numpy(), X=X)

    # Predict counterfactual treatment effects (CATE) across sample X
    cate_raw = model.effect(X)
    cate = np.asarray(cate_raw, dtype=np.float64).ravel()

    # Confidence intervals for CATE
    cate_lower_raw, cate_upper_raw = model.effect_interval(X, alpha=0.05)
    cate_lower = np.asarray(cate_lower_raw, dtype=np.float64).ravel()
    cate_upper = np.asarray(cate_upper_raw, dtype=np.float64).ravel()

    # Confidence interval for ATE
    ate = float(model.ate(X))
    ate_lower_raw, ate_upper_raw = model.ate_interval(X, alpha=0.05)
    ate_lower = float(ate_lower_raw)
    ate_upper = float(ate_upper_raw)

    causal_parameters = CausalParameters(
        cate=cate,
        cate_lower=cate_lower,
        cate_upper=cate_upper,
        ate=ate,
        ate_lower=ate_lower,
        ate_upper=ate_upper
        )

    return causal_parameters, propensity_scores

In [11]:
causal_parameters, propensity_scores = double_ml_econml_inline(
    df=df,
    feature_columns=feature_columns,
    treatment_column="treatment",
    outcome_column="y_factual"
    )

print_metrics("double_ml_econml", causal_parameters, true_cate=None)

NameError: name 'model_y' is not defined